# Experiment 7 · Stratified failure analysis (post-hoc)

This experiment involves **no training and no GPU**. It is a post-hoc breakdown of segmentation metrics that were already computed elsewhere: the **box-conditioned** full-supervised CNN baselines (U-Net, TransUNet, nnU-Net) that feed Table 1, and the LoRA foundation models (SAM2, MedSAM, MedSAM-2, SAM3) at the **5% few-shot** fraction and at full supervision. Here we slice those metrics by clinical and demographic strata — **TI-RADS level, nodule size, patient age, and sex** — to see where each model is strong or weak. Only **ThyroidXL** is stratified: it is the single cohort with enough test patients (n=739) to fill the bins; the other datasets have too few cases per stratum. The pipeline builds a unified long-format table, computes cluster-bootstrap confidence intervals and paired Wilcoxon tests per bin, and produces the per-bin data behind Figure 3.

> The per-image inputs for the box-conditioned CNNs and the LoRA foundation models carry gated (ThyroidXL) image filenames, so they are not shipped here. Regenerate them by running the box-conditioned CNN and foundation evaluations on your locally placed data, then re-run the cells below to rebuild the aggregate per-bin tables.

In [ ]:
# Move to the repository root so the experiment scripts resolve their paths.
import os
from pathlib import Path

cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "thyroidbench").is_dir()), cwd)
os.chdir(root)
print("Repo root:", root)

In [ ]:
# Check the inputs this experiment needs before doing anything slow. A missing
# dataset here means an unrun (or unplaced) setup step, not a bug in the experiment.
from pathlib import Path

REQUIRED = ['ddti', 'tn3k', 'thyroidxl', 'stanford_aimi']
GATED = {'thyroidxl', 'stanford_aimi'}

missing = [d for d in REQUIRED
           if not (Path('data/processed') / d / 'images').is_dir()
           or not (Path('data/splits') / f'{d}_test.csv').exists()]
if missing:
    print('Missing preprocessed data or splits for:', ', '.join(missing))
    for d in missing:
        if d in GATED:
            print(f'  {d:14s} access-gated -> place your approved copy first, see '
                  f'00_setup/01_place_gated_datasets.ipynb')
        else:
            print(f'  {d:14s} open -> download it with 00_setup/00_get_open_datasets.ipynb')
    print('Then run 00_setup/02_preprocess.ipynb and 00_setup/03_make_splits.ipynb.')
    print('\nYou can still run this experiment on whichever datasets ARE present '
          'by restricting the --dataset argument below.')
else:
    print('All four datasets are preprocessed and split.')


## Build the stratifier table

In [ ]:
!python experiments/exp7_stratified/build_stratifier_table.py

## Analyze (per-bin bootstrap CIs and paired Wilcoxon)

In [ ]:
!python experiments/exp7_stratified/analyze_stratified.py

## Results

In [ ]:
import pandas as pd
from pathlib import Path

results_dir = Path("experiments/exp7_stratified/results")
csvs = sorted(results_dir.rglob("*.csv"))
print("CSV files under", results_dir, ":")
for c in csvs:
    print("  ", c)

# Load a representative table: per-bin paired-Wilcoxon statistics.
stats_csv = next(c for c in csvs if c.name == "stats_per_bin.csv")
df = pd.read_csv(stats_csv)
print("\n", stats_csv, "shape:", df.shape)
df.head(20)